# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [46]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [47]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [48]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [49]:
# TODO: Load environment variables
# load_dotenv()
from pathlib import Path
load_dotenv(Path.cwd().parent.parent / "config.env")

True

### VectorDB Instance

In [50]:
chroma_client = chromadb.PersistentClient(
    path="chroma_db"
)

In [51]:
import chromadb

print(chromadb.__version__)

1.5.9


### Collection

In [52]:
import os

print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY"))
print("CHROMA_OPENAI_API_KEY:", os.getenv("CHROMA_OPENAI_API_KEY"))
print("TAVILY_API_KEY:", os.getenv("TAVILY_API_KEY"))

OPENAI_API_KEY: voc-128212795616886549858456a535e4d49dca0.66072079
CHROMA_OPENAI_API_KEY: voc-128212795616886549858456a535e4d49dca0.66072079
TAVILY_API_KEY: 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


In [53]:
import os

print(os.getcwd())

d:\UdaPlay-AI-Research-Agent\project\starter


In [54]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
# embedding_fn = embedding_functions.OpenAIEmbeddingFunction()
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-3-large"
)

try:
    chroma_client.delete_collection("udaplay")
    print("Old collection deleted.")
except Exception:
    print("Collection did not exist.")

collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

Old collection deleted.


In [55]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [56]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"""
    Game: {game['Name']}
    Platform: {game['Platform']}
    Genre: {game['Genre']}
    Publisher: {game['Publisher']}
    Release Year: {game['YearOfRelease']}
    Description: {game['Description']}
    """.strip()
    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [57]:
results = collection.query(
    query_texts=["Who developed FIFA 21?"],
    n_results=3
)

print(results)

{'ids': [['015', '013', '005']], 'embeddings': None, 'documents': [["Game: Halo Infinite\n    Platform: Xbox Series X|S\n    Genre: First-person shooter\n    Publisher: Xbox Game Studios\n    Release Year: 2021\n    Description: The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting.", 'Game: Kinect Adventures!\n    Platform: Xbox 360\n    Genre: Party\n    Publisher: Microsoft Game Studios\n    Release Year: 2010\n    Description: A collection of mini-games designed to showcase the capabilities of the Kinect motion sensor.', "Game: Marvel's Spider-Man 2\n    Platform: PlayStation 5\n    Genre: Action-adventure\n    Publisher: Sony Interactive Entertainment\n    Release Year: 2023\n    Description: The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters."]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'Description': "The lates

In [58]:
import os
import json

for file_name in sorted(os.listdir("games")):
    with open(os.path.join("games", file_name), "r", encoding="utf-8") as f:
        game = json.load(f)
        print(game["Name"])

Gran Turismo
Grand Theft Auto: San Andreas
Gran Turismo 5
Marvel's Spider-Man
Marvel's Spider-Man 2
Pokémon Gold and Silver
Pokémon Ruby and Sapphire
Super Mario World
Super Mario 64
Super Smash Bros. Melee
Wii Sports
Mario Kart 8 Deluxe
Kinect Adventures!
Minecraft
Halo Infinite


In [59]:
results = collection.query(
    query_texts=["Tell me about Minecraft"],
    n_results=1
)

print(results["documents"][0][0])
print(results["metadatas"][0][0])

Game: Minecraft
    Platform: Xbox One
    Genre: Sandbox, Survival
    Publisher: Mojang Studios
    Release Year: 2014
    Description: A sandbox game that allows players to build and explore infinite worlds, fostering creativity and adventure.
{'Name': 'Minecraft', 'YearOfRelease': 2014, 'Genre': 'Sandbox, Survival', 'Platform': 'Xbox One', 'Publisher': 'Mojang Studios', 'Description': 'A sandbox game that allows players to build and explore infinite worlds, fostering creativity and adventure.'}


In [60]:
print("Number of documents:", collection.count())

Number of documents: 15
